# Strategy tests — no API key needed

Offline checks for the tail-basket long straddle in `strategy.ipynb` / the main notebook. 
All cells run on stub data; `test_4` reads the main notebook file for structural checks.

## Test 1 · `strategy_pnl`: straddle profits both tails, bleeds when flat

In [1]:
def strategy_pnl(m_e, m_x, S_e, S_x, otm):
    dS = S_x - S_e
    dC_U = m_x[f"C_U{otm}"] - m_e[f"C_U{otm}"]
    dP_L = m_x[f"P_L{otm}"] - m_e[f"P_L{otm}"]
    dC_K = m_x["C_K"] - m_e["C_K"]
    dP_K = m_x["P_K"] - m_e["P_K"]
    return {"stock": dS / S_e, "long_call": dC_K / S_e,
            "covered_call": (dS - dC_U) / S_e, "protective_put": (dS + dP_L) / S_e,
            "collar": (dS + dP_L - dC_U) / S_e, "cash_secured_put": (-dP_L) / S_e,
            "long_straddle": (dC_K + dP_K) / S_e}

m_e = {"C_K": 5.0, "P_K": 5.0, "C_U0.05": 2.0, "P_L0.05": 2.0}
r_up = strategy_pnl(m_e, {"C_K": 13.0, "P_K": 2.0, "C_U0.05": 7.0, "P_L0.05": 0.5}, 100.0, 110.0, 0.05)
r_dn = strategy_pnl(m_e, {"C_K": 1.0, "P_K": 14.0, "C_U0.05": 0.2, "P_L0.05": 9.0}, 100.0, 90.0, 0.05)
r_flat = strategy_pnl(m_e, {"C_K": 3.0, "P_K": 3.0, "C_U0.05": 1.0, "P_L0.05": 1.0}, 100.0, 100.0, 0.05)
assert abs(r_up["long_straddle"] - 0.05) < 1e-9, r_up
assert abs(r_dn["long_straddle"] - 0.05) < 1e-9, r_dn
assert r_flat["long_straddle"] < 0, r_flat
print(f"up {r_up['long_straddle']:+.2%} / down {r_dn['long_straddle']:+.2%} / flat {r_flat['long_straddle']:+.2%} — PASS")


up +5.00% / down +5.00% / flat -4.00% — PASS


## Test 2 · Basket union: one row per filer per filing date, tags combined

In [2]:
import pandas as pd
raw = pd.DataFrame([
 {"cik": "1", "filing_date": pd.Timestamp("2024-02-21"), "tickers": ["UNH"],
  "accession_number": "a1", "filing_url": "u1", "supporting_text": "breach",
  "tertiary_category": "cybersecurity_incident"},
 {"cik": "1", "filing_date": pd.Timestamp("2024-02-21"), "tickers": ["UNH"],
  "accession_number": "a1", "filing_url": "u1", "supporting_text": "breach",
  "tertiary_category": "material_litigation"},
 {"cik": "2", "filing_date": pd.Timestamp("2024-07-11"), "tickers": ["T"],
  "accession_number": "a2", "filing_url": "u2", "supporting_text": "suit",
  "tertiary_category": "material_litigation"},
])
ex = raw.explode("tickers").rename(columns={"tickers": "ticker"})
ev = (ex.sort_values(["cik", "filing_date"])
        .groupby(["cik", "filing_date"], as_index=False)
        .agg(ticker=("ticker", "first"), accession_number=("accession_number", "first"),
             filing_url=("filing_url", "first"), supporting_text=("supporting_text", "first"),
             tags=("tertiary_category", lambda s: "+".join(sorted(set(s))))))
assert len(ev) == 2, ev
assert ev.loc[0, "tags"] == "cybersecurity_incident+material_litigation", ev
assert ev.loc[1, "tags"] == "material_litigation", ev
print(ev[["ticker", "filing_date", "tags"]].to_string(), "\nUNION PASS")


  ticker filing_date                                        tags
0    UNH  2024-02-21  cybersecurity_incident+material_litigation
1      T  2024-07-11                         material_litigation 
UNION PASS


## Test 3 · Display guard: `sample(min(5, len))` never over-samples

In [3]:
import pandas as pd
for n in (0, 1, 4, 5, 9):
    df = pd.DataFrame({"a": range(n)})
    got = df.sample(min(5, len(df)), random_state=1)
    assert len(got) == min(5, n), (n, len(got))
print("n = 0/1/4/5/9 all sample safely — PASS")


n = 0/1/4/5/9 all sample safely — PASS


## Test 4 · Main-notebook structural check (reads the challenge file, no API)

In [4]:
import ast, json
nb = json.load(open("gator-quant-hacks-8k-options-challenge.ipynb", encoding="utf-8"))
full = "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
probes = ['TAIL_TAGS', 'FORBEARANCE_TAGS', '(dC_K + dP_K) / S_e',
          '["C_K", "P_K"]', 'min(5, len(events))', 'str | list[str]',
          'lambda s: "+".join(sorted(set(s)))']
for probe in probes:
    assert probe in full, f"MISSING: {probe}"
    print(f"found: {probe[:60]}")
fails = []
for c in nb["cells"]:
    s = "".join(c["source"]) if c["cell_type"] == "code" else ""
    if not s.strip() or any(l.strip().startswith("%") for l in s.splitlines()):
        continue  # skip IPython magics (e.g. the optional %pip cell)
    try:
        ast.parse(s)
    except SyntaxError as e:
        fails.append(str(e))
assert not fails, fails
print(len(nb["cells"]), "cells, all parse -- PASS")


found: TAIL_TAGS
found: FORBEARANCE_TAGS
found: (dC_K + dP_K) / S_e
found: ["C_K", "P_K"]
found: min(5, len(events))
found: str | list[str]
found: lambda s: "+".join(sorted(set(s)))
45 cells, all parse -- PASS


## Test 5 · Convertible-strategy bugfix regressions (reads `strategy.ipynb`, no API)

In [ ]:
import ast, json
nb = json.load(open("strategy.ipynb", encoding="utf-8"))
full = "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
probes = ['MAX_STALE_SESSIONS', 'after dedupe', 'WARNING', 'def stock(self',
          '"high": [float', 'S_e = pe.stock(e_day)', 'has_capped_call: bool = False',
          'bool(getattr(row, "has_capped_call", False))']
for probe in probes:
    assert probe in full, f"MISSING: {probe}"
    print(f"found: {probe[:50]}")
import pandas as pd, numpy as np
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, nearest_workday,
    USMartinLutherKingJr, USPresidentsDay, GoodFriday, USMemorialDay, USLaborDay, USThanksgivingDay)
from pandas.tseries.offsets import CustomBusinessDay
from dataclasses import dataclass
cal_src = next("".join(c["source"]) for c in nb["cells"]
               if c["cell_type"] == "code" and "class NYSEHolidays" in "".join(c["source"]))
leg_src = next("".join(c["source"]) for c in nb["cells"]
               if c["cell_type"] == "code" and "class Leg:" in "".join(c["source"]))
ns = {"pd": pd, "np": np, "AbstractHolidayCalendar": AbstractHolidayCalendar, "Holiday": Holiday,
      "nearest_workday": nearest_workday, "USMartinLutherKingJr": USMartinLutherKingJr,
      "USPresidentsDay": USPresidentsDay, "GoodFriday": GoodFriday, "USMemorialDay": USMemorialDay,
      "USLaborDay": USLaborDay, "USThanksgivingDay": USThanksgivingDay,
"CustomBusinessDay": CustomBusinessDay, "dataclass": dataclass,
"MAX_STALE_SESSIONS": 3}
exec(cal_src, ns)
import re
m = re.search(r"class Leg:.*?def volume_on.*?return float\(self\.bars\[\"volume\"\]\.get\(pd\.Timestamp\(day\), 0\.0\)\)", leg_src, re.S)
exec("@dataclass\n" + m.group(0), ns)
fresh = pd.DataFrame({"close": [1.0, 1.1], "volume": [10, 12]}, index=pd.DatetimeIndex(
    [pd.Timestamp("2025-05-19"), pd.Timestamp("2025-05-20")], name="session"))
assert ns["Leg"]("t", "put", 87.0, fresh).mark(pd.Timestamp("2025-05-23")) == 1.1
assert np.isnan(ns["Leg"]("t", "put", 87.0, fresh).mark(pd.Timestamp("2025-05-27")))
print("staleness-guard regression PASS")


## Test 6 · Cash-out exits + leverage accounting (reads `strategy.ipynb`, no API)

In [ ]:
import json
nb = json.load(open("strategy.ipynb", encoding="utf-8"))
full = "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
for probe in ['LEVERAGE = 20.0', 'def apply_cashout_exits', 'exit_pnl', 'max_loss_1x', 'FULL-WIDTH max loss']:
    assert probe in full, f"MISSING: {probe}"
import pandas as pd, numpy as np
fn_src = next("".join(c["source"]) for c in nb["cells"]
              if c["cell_type"] == "code" and "def apply_cashout_exits" in "".join(c["source"]))
ns = {"pd": pd, "np": np, "EXIT_DAY_LIMIT": 10}
ns["STRATEGIES"] = ["stock", "bull_put_spread", "bear_put_spread"]
exec(fn_src[:fn_src.index("def evaluate(")], ns)
rows = []
for h, sh, p in [(0, 0, 0.0), (1, 1, 0.05), (3, 3, 0.16), (10, 10, 0.21)]:
    rows.append({'ticker': 'T', 'event_date': pd.Timestamp('2025-01-01'), 'entry': 'post',
                 'otm': 0.0, 'horizon': h, 'sessions_held': sh, 'bull_put_spread': p, 'credit_pre': 0.30})
r = ns["apply_cashout_exits"](pd.DataFrame(rows), "bull_put_spread").iloc[0]
assert r["exit_horizon_bull_put_spread"] == 3 \
    and bool(r["exited_half_bull_put_spread"]) \
    and abs(r["exit_pnl_bull_put_spread"] - 0.16) < 1e-9, r
LEV = 20.0
assert abs(((0.0021 - 0.0040) * LEV) - (-0.038)) < 1e-3
print("cash-out + leverage accounting PASS")


## Test 7 · Inverted strategy: bear mirror + exits (reads `strategy.ipynb`, no API)

In [ ]:
import json
nb = json.load(open("strategy.ipynb", encoding="utf-8"))
full = "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
for probe in ['INVERT_TO_BEAR', '"bear_put_spread": -spread_pnl',
                'WINNER = "bear_put_spread" if INVERT_TO_BEAR', 'exit_pnl_{WINNER}']:
    assert probe in full, f"MISSING: {probe}"
import pandas as pd, numpy as np
eng = next("".join(c["source"]) for c in nb["cells"]
           if c["cell_type"] == "code" and "def strategy_pnl" in "".join(c["source"]))
ns = {"pd": pd, "np": np, "EXIT_DAY_LIMIT": 10}
exec(eng[:eng.index("def evaluate(")], ns)
r = ns["strategy_pnl"]({"short_put": 2.0, "long_put": 1.0}, {"short_put": 1.0, "long_put": 0.8}, 100.0, 99.0, 0.0)
assert abs(r["bear_put_spread"] + r["bull_put_spread"]) < 1e-12, r
rows = []
for h, sh, pb in [(0, 0, 0.0), (3, 3, -0.006), (5, 5, -0.002)]:
    rows.append({'ticker': 'SO', 'event_date': pd.Timestamp('2025-05-21'), 'entry': 'post', 'otm': 0.0,
                 'horizon': h, 'sessions_held': sh, 'short_strike': 88.0, 'long_strike': 87.0,
                 'S_entry': 88.0, 'credit_pre': 0.31 / 88.0, 'bull_put_spread': pb, 'bear_put_spread': -pb})
out = ns["apply_cashout_exits"](pd.DataFrame(rows), "bear_put_spread").iloc[0]
assert out["exit_horizon_bear_put_spread"] == 3 and bool(out["exited_half_bear_put_spread"])
assert abs(out["exit_pnl_bear_put_spread"] - 0.006) < 1e-9, out
print("bear mirror + exits PASS")


## Test 7 · Inverted strategy: bear mirror + exits (reads `strategy.ipynb`, no API)

In [ ]:
import json
nb = json.load(open("strategy.ipynb", encoding="utf-8"))
full = "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
for probe in ['INVERT_TO_BEAR', '"bear_put_spread": -spread_pnl',
                'WINNER = "bear_put_spread" if INVERT_TO_BEAR', 'exit_pnl_{WINNER}']:
    assert probe in full, f"MISSING: {probe}"
import pandas as pd, numpy as np
eng = next("".join(c["source"]) for c in nb["cells"]
           if c["cell_type"] == "code" and "def strategy_pnl" in "".join(c["source"]))
ns = {"pd": pd, "np": np, "EXIT_DAY_LIMIT": 10}
exec(eng[:eng.index("def evaluate(")], ns)
r = ns["strategy_pnl"]({"short_put": 2.0, "long_put": 1.0}, {"short_put": 1.0, "long_put": 0.8}, 100.0, 99.0, 0.0)
assert abs(r["bear_put_spread"] + r["bull_put_spread"]) < 1e-12, r
rows = []
for h, sh, pb in [(0, 0, 0.0), (3, 3, -0.006), (5, 5, -0.002)]:
    rows.append({'ticker': 'SO', 'event_date': pd.Timestamp('2025-05-21'), 'entry': 'post', 'otm': 0.0,
                 'horizon': h, 'sessions_held': sh, 'short_strike': 88.0, 'long_strike': 87.0,
                 'S_entry': 88.0, 'credit_pre': 0.31 / 88.0, 'bull_put_spread': pb, 'bear_put_spread': -pb})
out = ns["apply_cashout_exits"](pd.DataFrame(rows), "bear_put_spread").iloc[0]
assert out["exit_horizon_bear_put_spread"] == 3 and bool(out["exited_half_bear_put_spread"])
assert abs(out["exit_pnl_bear_put_spread"] - 0.006) < 1e-9, out
print("bear mirror + exits PASS")


## Test 7 · Inverted strategy: bear mirror + exits (reads `strategy.ipynb`, no API)

In [ ]:
import json
nb = json.load(open("strategy.ipynb", encoding="utf-8"))
full = "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
for probe in ['INVERT_TO_BEAR', '"bear_put_spread": -spread_pnl',
                'WINNER = "bear_put_spread" if INVERT_TO_BEAR', 'exit_pnl_{WINNER}']:
    assert probe in full, f"MISSING: {probe}"
import pandas as pd, numpy as np
eng = next("".join(c["source"]) for c in nb["cells"]
           if c["cell_type"] == "code" and "def strategy_pnl" in "".join(c["source"]))
ns = {"pd": pd, "np": np, "EXIT_DAY_LIMIT": 10}
exec(eng[:eng.index("def evaluate(")], ns)
r = ns["strategy_pnl"]({"short_put": 2.0, "long_put": 1.0}, {"short_put": 1.0, "long_put": 0.8}, 100.0, 99.0, 0.0)
assert abs(r["bear_put_spread"] + r["bull_put_spread"]) < 1e-12, r
rows = []
for h, sh, pb in [(0, 0, 0.0), (3, 3, -0.006), (5, 5, -0.002)]:
    rows.append({'ticker': 'SO', 'event_date': pd.Timestamp('2025-05-21'), 'entry': 'post', 'otm': 0.0,
                 'horizon': h, 'sessions_held': sh, 'short_strike': 88.0, 'long_strike': 87.0,
                 'S_entry': 88.0, 'credit_pre': 0.31 / 88.0, 'bull_put_spread': pb, 'bear_put_spread': -pb})
out = ns["apply_cashout_exits"](pd.DataFrame(rows), "bear_put_spread").iloc[0]
assert out["exit_horizon_bear_put_spread"] == 3 and bool(out["exited_half_bear_put_spread"])
assert abs(out["exit_pnl_bear_put_spread"] - 0.006) < 1e-9, out
print("bear mirror + exits PASS")


## Test 8 · Rescue: true placebo, liquidity screen, OOS (reads `strategy.ipynb`, no API)

In [ ]:
import json
nb = json.load(open("strategy.ipynb", encoding="utf-8"))
full = "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
for probe in ['MIN_ENTRY_VOLUME', 'def filter_liquid', 'def sample_placebo_events',
                'difference_board(results, pres)', 'oos = run_study', 'mandatory convertible']:
    assert probe in full, f"MISSING: {probe}"
assert "max_events=N_PLACEBO" not in full, "fake placebo still present"
import pandas as pd, numpy as np
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, nearest_workday,
    USMartinLutherKingJr, USPresidentsDay, GoodFriday, USMemorialDay, USLaborDay, USThanksgivingDay)
from pandas.tseries.offsets import CustomBusinessDay
ns = {"pd": pd, "np": np, "AbstractHolidayCalendar": AbstractHolidayCalendar, "Holiday": Holiday,
      "nearest_workday": nearest_workday, "USMartinLutherKingJr": USMartinLutherKingJr,
      "USPresidentsDay": USPresidentsDay, "GoodFriday": GoodFriday, "USMemorialDay": USMemorialDay,
      "USLaborDay": USLaborDay, "USThanksgivingDay": USThanksgivingDay,
      "CustomBusinessDay": CustomBusinessDay, "PLACEBO_GAP_DAYS": 30, "ATR_PERIOD": 14,
      "MIN_ENTRY_VOLUME": 20}
cal_src = next("".join(c["source"]) for c in nb["cells"]
               if c["cell_type"] == "code" and "class NYSEHolidays" in "".join(c["source"]))
exec(cal_src, ns)
ns["LAST_SESSION"] = ns["CAL"][-1]
rs_src = next("".join(c["source"]) for c in nb["cells"]
              if c["cell_type"] == "code" and "def sample_placebo_events" in "".join(c["source"]))
exec(rs_src[rs_src.index("def sample_placebo_events"):rs_src.index("if RUN_PLACEBO")], ns)
ev = pd.DataFrame({"ticker": ["SO", "T"],
                   "filing_date": pd.to_datetime(["2025-05-21", "2024-07-11"])})
peb = ns["sample_placebo_events"](ev, 8, "2025-01-01", "2025-12-31")
assert len(peb) > 0 and "has_capped_call" in peb.columns
print("true-placebo sampler PASS")


## Test 9 · Capped-call detector breadth (reads `strategy.ipynb`, no API)

In [ ]:
import json
nb = json.load(open("strategy.ipynb", encoding="utf-8"))
src = next("".join(c["source"]) for c in nb["cells"]
           if c["cell_type"] == "code" and "def has_capped_call" in "".join(c["source"]))
assert "CAPPED_CALL_PHRASES" in src
ns = {}
exec(src[src.index("CAPPED_CALL_PHRASES"):src.index("def build_convertible_events")], ns)
f = ns["has_capped_call"]
positives = ["capped call transactions", "call spread to hedge dilution",
             "convertible note hedge and warrant", "bond hedge"]
negatives = ["issued convertible senior notes", "securities purchase agreement", ""]
assert all(f({"supporting_text": t}) for t in positives)
assert not any(f({"supporting_text": t}) for t in negatives)
print(f"detector breadth PASS ({len(ns["CAPPED_CALL_PHRASES"])} phrases)")
